In [1]:
# Data Cleaning - Adzuna Job Postings
# cleaning and normalising the job postings for NLP and classifier use later

import pandas as pd

# loading the raw file
df = pd.read_csv('Adzuna_Postings_AllSectors_13Jun2026.csv')

print("shape:", df.shape)
print("\ncolumns:", df.columns.tolist())
print("\nfirst 3 rows:")
print(df.head(3))

shape: (1250, 9)

columns: ['sector', 'title', 'company', 'location', 'salary_min', 'salary_max', 'description', 'created', 'redirect_url']

first 3 rows:
       sector                       title                          company  \
0  Technology  Trainee Software Developer                     ITOL Recruit   
1  Technology          Software Developer         Woodside Logistics Group   
2  Technology   Senior Software Developer  Spectrum It Recruitment Limited   

                   location  salary_min  salary_max  \
0                        UK    30000.00    45000.00   
1  Ballyclare, Newtownabbey    47709.66    47709.66   
2    Southampton, Hampshire    70000.00    70000.00   

                                         description               created  \
0  Trainee Coding Programme – No Experience Neede...  2026-06-01T17:30:25Z   
1  Job Title: Software Developer Responsible to: ...  2026-06-11T20:56:13Z   
2  Senior Software Developer Hybrid - 2 days per ...  2026-06-11T04:09:25Z   

In [2]:
# cleaning salary fields
# adding a midpoint salary column
df['salary_mid'] = (df['salary_min'] + df['salary_max']) / 2

# removing unrealistic salaries - anything under 10k or over 300k
df_clean = df[(df['salary_mid'] >= 10000) & (df['salary_mid'] <= 300000)].copy()

print("rows before salary filter:", len(df))
print("rows after salary filter:", len(df_clean))

# checking salary stats per sector
print("\nmedian salary by sector:")
print(df_clean.groupby('sector')['salary_mid'].median().round(0))

rows before salary filter: 1250
rows after salary filter: 1248

median salary by sector:
sector
Education      50960.0
Engineering    48700.0
Finance        58537.0
Healthcare     44470.0
Technology     52500.0
Name: salary_mid, dtype: float64


In [3]:
# cleaning company names - stripping extra spaces
df_clean['company'] = df_clean['company'].str.strip().str.title()

# cleaning location - extracting just the city part before the comma
df_clean['city'] = df_clean['location'].apply(
    lambda x: str(x).split(',')[0].strip() if ',' in str(x) else str(x).strip()
)

# cleaning date - keeping just the date part not the time
df_clean['date_posted'] = pd.to_datetime(df_clean['created']).dt.date

# dropping columns i dont need
df_clean = df_clean.drop(columns=['redirect_url', 'created', 'location'])

print("shape after cleaning:", df_clean.shape)
print("\ncolumns:", df_clean.columns.tolist())
print("\nfirst 3 rows:")
print(df_clean.head(3))

shape after cleaning: (1248, 9)

columns: ['sector', 'title', 'company', 'salary_min', 'salary_max', 'description', 'salary_mid', 'city', 'date_posted']

first 3 rows:
       sector                       title                          company  \
0  Technology  Trainee Software Developer                     Itol Recruit   
1  Technology          Software Developer         Woodside Logistics Group   
2  Technology   Senior Software Developer  Spectrum It Recruitment Limited   

   salary_min  salary_max                                        description  \
0    30000.00    45000.00  Trainee Coding Programme – No Experience Neede...   
1    47709.66    47709.66  Job Title: Software Developer Responsible to: ...   
2    70000.00    70000.00  Senior Software Developer Hybrid - 2 days per ...   

   salary_mid         city date_posted  
0    37500.00           UK  2026-06-01  
1    47709.66   Ballyclare  2026-06-11  
2    70000.00  Southampton  2026-06-11  


In [4]:
# checking description length - important for NLP later
df_clean['desc_length'] = df_clean['description'].str.len()
print("description length stats:")
print(df_clean['desc_length'].describe())

# flagging recruitment agencies - they appear a lot in the data
agencies = ['Hays', 'Reed', 'Adecco', 'Manpower', 'Randstad', 
            'Recruit', 'Recruitment', 'Staffing', 'Agency']
df_clean['is_agency'] = df_clean['company'].apply(
    lambda x: any(a.lower() in str(x).lower() for a in agencies)
)

print("\nagency vs direct employer:")
print(df_clean['is_agency'].value_counts())

description length stats:
count    1248.000000
mean      499.262019
std        15.050125
min       177.000000
25%       500.000000
50%       500.000000
75%       500.000000
max       500.000000
Name: desc_length, dtype: float64

agency vs direct employer:
is_agency
False    791
True     457
Name: count, dtype: int64


In [5]:
# saving clean file
df_clean.to_csv('Adzuna_Clean.csv', index=False)
print("saved! shape:", df_clean.shape)
print("\nsector counts:")
print(df_clean['sector'].value_counts())

saved! shape: (1248, 11)

sector counts:
sector
Healthcare     250
Finance        250
Engineering    250
Technology     249
Education      249
Name: count, dtype: int64


In [6]:
# notes for dissertation
print("key cleaning steps done:")
print("- removed 2 unrealistic salary outliers (under 10k or over 300k)")
print("- added salary midpoint column")
print("- extracted city from location field")
print("- standardised company names to title case")
print("- flagged 457 recruitment agency postings vs 791 direct employers")
print("- descriptions averaged 499 chars - good enough for NLP")
print("- Finance highest median salary at £58,537")
print("- saved as Adzuna_Clean.csv - 1,248 rows, 11 columns")

key cleaning steps done:
- removed 2 unrealistic salary outliers (under 10k or over 300k)
- added salary midpoint column
- extracted city from location field
- standardised company names to title case
- flagged 457 recruitment agency postings vs 791 direct employers
- descriptions averaged 499 chars - good enough for NLP
- Finance highest median salary at £58,537
- saved as Adzuna_Clean.csv - 1,248 rows, 11 columns
